# Extraction — current dev logic, integrated audit contract

Uses the original `extract.py` from the GitHub dev branch. Dry-run and schema settings are independent.

In [0]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "config" / "run.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "config" / "run.json").exists(), "Run notebook from repo root or immediate subfolder"
sys.path.insert(0, str(ROOT))
from common.runtime import load_run
from common.execute import sync_layer
run = load_run(ROOT / "config" / "run.json")

import uuid
from pyspark.sql import functions as F
from extraction import extract as extractor
from common.runtime import append_audit, spark_schema
from pyspark.sql.types import StructType, StructField, StringType, LongType

config = extractor.load_config(ROOT / "extraction" / "config.json")
audit, schema_changes = sync_layer(spark, ROOT, "audit", run)
print("Audit schema:", schema_changes)
batch_id = str(uuid.uuid4())
print("Batch:", batch_id)

In [0]:
events_table = f"{audit['catalog']}.audit.audit_events_tbl"
files_table = f"{audit['catalog']}.audit.audit_files_tbl"
# For a brand-new environment, create the approved tables. Existing tables are NEVER overwritten.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {audit['catalog']}.audit")
for table in ("audit_events_tbl", "audit_files_tbl"):
    full = f"{audit['catalog']}.audit.{table}"
    if not spark.catalog.tableExists(full):
        if not run["schema_change_requested"] or run["schema_dry_run"]:
            raise RuntimeError(f"{full} missing; enable approved schema deployment")
        spark.createDataFrame([], spark_schema(audit, table)).write.format("delta").saveAsTable(full)

In [0]:
discovered = extractor.discover_archives(config)
discovery_schema = StructType([
    StructField("zip_key", StringType()), StructField("source_zip_name", StringType()),
    StructField("source_url", StringType()), StructField("data_month", StringType()),
    StructField("zip_size_bytes", LongType())])
discovered_df = spark.createDataFrame(discovered, discovery_schema)
done = (spark.table(events_table)
        .filter((F.col("stage") == "UNZIP") & (F.col("status") == "SUCCESS"))
        .select("zip_key").distinct())
pending = discovered_df if config.get("force", False) else discovered_df.join(done, "zip_key", "left_anti")
if config["mode"] == "incremental":
    month = pending.agg(F.min("data_month").alias("month")).first()["month"]
    pending = pending.filter(F.col("data_month") == month) if month else pending.limit(0)
limit = int(config.get("max_files_per_run",24))
if pending.limit(limit+1).count() > limit:
    raise ValueError(f"More than {limit} pending ZIPs")
approved = [r.asDict() for r in pending.orderBy("data_month","source_zip_name").collect()]
print("Archives approved:", len(approved))

In [0]:
result = extractor.extract_archives(config, approved, batch_id)
print(result["status"], "events:", len(result["audit_events"]),
      "files:", len(result["audit_files"]))
if not config["dry_run"]:
    # Extraction file metadata is written before events, matching original notebook.
    append_audit(spark, result["audit_files"], audit, "audit_files_tbl")
    append_audit(spark, result["audit_events"], audit, "audit_events_tbl")
if result["status"] == "FAILED":
    raise RuntimeError(result["error_message"])